# Baseline
Den övergripande explorativa dataanalysen genomförs separat i projektets EDA 01_exploration. Den här notebooken fokuserar på att bygga och testa projektets baseline för kompetensidentifiering.

Baselinen utvecklas endast på träningsdata. Projektets test-set används
senare för slutlig utvärdering.

In [27]:
# läser in träningsannonser:
import json
import pandas as pd
import re

with open("../data/train_set.json", "r", encoding="utf-8") as file:
    ads = json.load(file)

df = pd.DataFrame(ads)

In [28]:
# liten EDA som är relevant för baseline:
df[["id", "title", "published", "region", "description"]].head()

,id,title,published,region,description
0,29556320,Business Consultant Dynamics 365 Supply Chain ...,2025-03-19T10:04:43,Stockholms län,Har du erfarenhet av Dynamics 365 och/eller AX...
1,31266797,Hälsoinformatiker för framtidens vårdinformation,2026-07-10T17:08:31,Östergötlands län,"Omvärlden förändras i snabb takt. Ny teknik, n..."
2,31129893,Data Scientist till Innovativt Bolag,2026-06-02T19:12:40,Västra Götalands län,Är du en analytisk och lösningsorienterad pers...
3,29479271,Dynamics 365 Finance Consultant,2025-02-19T12:38:38,NaN,Vill du arbeta med den senaste tekniken och hj...
4,31249545,"Enterprise arkitekt, Gävle/Uppsala/Stockholm",2026-07-06T01:35:55,Gävleborgs län,Lantmäteriet är den myndighet som kartlägger S...


In [29]:
# Enkel kontroll av datan, dubbelkollar saknade värden:
df["description"].isna().sum()

np.int64(0)

In [30]:
# dubbelkollar id-dubbletter:
df["id"].duplicated().sum()

np.int64(0)

In [31]:
# läser in kompetenserna för SSYK 2511
with open("../data/skills_2511.json", "r", encoding="utf-8") as file:
    skills_data = json.load(file)

In [32]:
# kontrolerar hur filen ser ut:
print("\nStruktur:",type(skills_data))
print("----------------------------")
skills_data[0]


Struktur: <class 'list'>
----------------------------


{'skill_id': '13ad_dCC_Tn6',
 'skill': 'OOS, styr- och utvecklingsmodell',
 'headlines': ['Styr- och utvecklingsmodeller']}

In [33]:
# bygger en lista med kompetensnamn som baseline kan matcha mot:
skill_names = [
    item["skill"]
    for item in skills_data
]

print("\nAntal kompetenser:", len(skill_names))
print("----------------------------")

print("\nExempel på 20 kompetenser:")
skills_df = pd.DataFrame(skills_data)
print(skills_df.head(20))
print("----------------------------")


Antal kompetenser: 204
----------------------------

Exempel på 20 kompetenser:
        skill_id                                              skill  \
0   13ad_dCC_Tn6                   OOS, styr- och utvecklingsmodell   
1   16ZN_3Zp_bYV                                Odoo, molnplattform   
2   19PK_uxU_Wzy                              Bower, pakethanterare   
3   1CmG_8hR_igc                                   Bluetooth-teknik   
4   1QYQ_Xka_TQ5      Microsoft Autopilot, systemutvecklingsverktyg   
5   2CXh_FUJ_vTH  Office SharePoint Server (MOSS), applikationsp...   
6   2rqG_WLp_em4                   Ledningssystem (ISO 9000-serien)   
7   2vCi_hQz_Y9o                                         WAP-teknik   
8   39C3_4Tm_3ud                       Asset Information Management   
9   3H7m_pt8_7b2                HTTP/HTTPS, kommunikationsprotokoll   
10  3JPR_oR7_UUS      Sun Java Enterprise System, webbserverprogram   
11  3KME_Qwm_D4V                                IBM Integration Bus

### Första enkla versionen: naive baseline

In [34]:
# baseline funktionen:
def naive_baseline_extract_skills(text, skills):
    found_skills = []

    text = text.lower()

    for skill in skills:
        if skill.lower() in text:
            found_skills.append(skill)

    return found_skills

In [35]:
# testar baseline på några annonser:
sample_df = df.head(20).copy()

sample_df["naive_baseline_skills"] = sample_df["description"].apply(
    lambda text: naive_baseline_extract_skills(text, skill_names)
)

In [36]:
sample_df[
    ["title", "naive_baseline_skills"]
]

,title,naive_baseline_skills
0,Business Consultant Dynamics 365 Supply Chain ...,[]
1,Hälsoinformatiker för framtidens vårdinformation,"[Informationsmodellering, Förändringsledning, ..."
2,Data Scientist till Innovativt Bolag,[REST]
3,Dynamics 365 Finance Consultant,[]
4,"Enterprise arkitekt, Gävle/Uppsala/Stockholm",[]
5,IT-konsult,[]
6,Senior konsult inom D365 Finance & Operations ...,[]
7,Senior OpenStack-arkitekt,[REST]
8,Lösningsarkitekt,[Webbutveckling]
9,Senior system architect,[]


Baseline fungerar tekniskt, men många annonser får inga träffar. Det kan bero på att:
- taxonomin inte innehåller exakt samma formulering som annonsen
- annonsen skriver en förkortning, t.ex. ML istället för Machine Learning
- kompetensen uttrycks på engelska/svenska på annat sätt
- kompetensen finns implicit men inte som exakt fras
- SSYK 2511-listan inte täcker exakt allt som står i varje annons

In [37]:
# undersöker annonser med tomma träffar:
empty_matches = sample_df[
    sample_df["naive_baseline_skills"].apply(len) == 0
]

sample_errors = empty_matches.head(5)

In [38]:
# annonser med empty matches:
sample_errors[
    ["title", "description"]
]

,title,description
0,Business Consultant Dynamics 365 Supply Chain ...,Har du erfarenhet av Dynamics 365 och/eller AX...
3,Dynamics 365 Finance Consultant,Vill du arbeta med den senaste tekniken och hj...
4,"Enterprise arkitekt, Gävle/Uppsala/Stockholm",Lantmäteriet är den myndighet som kartlägger S...
5,IT-konsult,Om jobbet\nCapgemini strävar efter att vara en...
6,Senior konsult inom D365 Finance & Operations ...,Vill du verka som Functional Solution Architec...


In [39]:
# Felanalys som vi kan granska manuellt:
for i, row in sample_errors.iterrows():
    print("\n" + "=" * 80)
    print("TITLE:", row["title"])
    print("NAIVE BASELINE:", row["naive_baseline_skills"])
    
    print("\nDESCRIPTION:")
    print(row["description"])


TITLE: Business Consultant Dynamics 365 Supply Chain Management – Stockholm
NAIVE BASELINE: []

DESCRIPTION:
Har du erfarenhet av Dynamics 365 och/eller AX2012, särskilt inom supply chain management? Vill du hjälpa våra kunder att maximera potentialen i deras D365-lösningar genom att utmana och vägleda dem? Om du är en duktig kommunikatör som vill vara en trusted digital advisor för dina kunder, kan du vara den vi söker  
Vad kommer du att göra?I den här rollen kommer du att spela en nyckelroll i att hjälpa våra kunder att maximera potentialen i sina Dynamics 365-lösningar. Du kommer att arbeta nära kunderna för att bygga långsiktiga relationer, förstå deras unika behov och koppla dessa till tekniska möjligheter. Din expertis blir avgörande för att kunderna ska kunna dra full nytta av Microsofts senaste releaser och implementera skräddarsydda lösningar som gör skillnad. Din roll sträcker sig över hela implementationsprocessen – från rådgivning och förfrågningar till att hantera föränd

Den enkla baselinen gav inga träffar i flera av annonserna. Vid manuell
granskning innehöll annonserna framför allt generella beskrivningar av
IT-arbete, projektledning, systemutveckling och konsultarbete, erfarenhet av att arbeta i projekt, erfarenhet att arbeta och samverka med olika typer av intressenter - utvecklare, testare men inga tydliga kompetenser som matchade kompetenslistan exakt.

Resultatet bedömmer vi därför som rimligt för dessa exempel.

### Naive baseline på hela träningsdatasetet:

In [40]:
# sammanställer resultatet på hela datasetet och inte endast 20 annonser:

df["naive_baseline_skills"] = df["description"].apply(
    lambda text: naive_baseline_extract_skills(text, skill_names)
)

all_naive_skills = df["naive_baseline_skills"].explode()

naive_skill_counts = all_naive_skills.value_counts()

naive_skill_counts.head(20)

naive_baseline_skills
REST                       798
Informationsarkitektur     136
Förändringsledning         130
Datamodellering            119
Informationsmodellering     92
Informatik                  80
Masterdata                  79
Systemförvaltning           76
Agila arbetsmetoder         54
Verksamhetsanalys           42
API-design                  17
Projektledning, IT          15
Nätverksteknik              14
Prediktiv analys            11
Molnteknik                  10
Datawarehouse                9
Webbutveckling               8
Interaktionsdesign           6
Databasdesign                4
Budgetansvar                 4
Name: count, dtype: int64

`REST = 798` betyder att baseline hittade termen REST i 798 annonser, men vad är REST för något ?

In [41]:
# undersöker i vilka annonser förekommer REST: 
rest_ads = df[
    df["naive_baseline_skills"].apply(lambda x: "REST" in x)
]

rest_ads[["title", "description"]].head()

,title,description
2,Data Scientist till Innovativt Bolag,Är du en analytisk och lösningsorienterad pers...
7,Senior OpenStack-arkitekt,Om företaget\nHos Avaron får du tryggheten i e...
18,Senior Cloud Architect - MS Entra ID,"I rollen blir du expert på att designa, optime..."
27,Solutions Architect till Husqvarna Group!,Som Lösningsarkitekt på Husqvarna Group får du...
34,Senior Kravanalytiker,Senior Kravanalytiker – Produktutveckling inom...


In [42]:
# kollar upp var exakt fångar baseline ordet REST:
row = rest_ads.iloc[0]

print("TITLE:", row["title"])

text = row["description"].lower()

for match in re.finditer("rest", text):
    start = match.start()

    print(
        text[max(0, start - 50): start + 50]
    )
    print("-" * 80)

TITLE: Data Scientist till Innovativt Bolag
et av modeller och kontinuerligt förbättra deras prestanda
delta i hela produktutvecklingsprocessen 
--------------------------------------------------------------------------------


In [43]:
# letar genom fler :
for _, row in rest_ads.head(20).iterrows():

    text = row["description"].lower()

    for match in re.finditer("rest", text):
        start = match.start()

        print("\nTITLE:", row["title"])
        print(
            text[max(0, start - 50): start + 50]
        )
        print("-" * 80)


TITLE: Data Scientist till Innovativt Bolag
et av modeller och kontinuerligt förbättra deras prestanda
delta i hela produktutvecklingsprocessen 
--------------------------------------------------------------------------------

TITLE: Senior OpenStack-arkitekt
ingsregler och interna riktlinjer. du analyserar prestanda och kapacitet samt driver förbättringar s
--------------------------------------------------------------------------------

TITLE: Senior Cloud Architect - MS Entra ID
pande iam-strategier för att säkerställa optimal prestanda, skalbarhet och säkerhet i azure-miljön. 
--------------------------------------------------------------------------------

TITLE: Solutions Architect till Husqvarna Group!
 för professionella användare. de för samman hög prestanda, användarvänlighet och säkerhet och erbju
--------------------------------------------------------------------------------

TITLE: Senior Kravanalytiker
ionella och icke-funktionella krav kopplade till prestanda, användar

Den första implementationen använde enkel substring-matchning. Det innebär
att en kompetens kan identifieras även när tecknen endast förekommer som en
del av ett annat ord (exempel: pRESTanda, pRESTigelös).

Ett tydligt exempel var kompetensen `REST`, som fick ett mycket stort antal
träffar. Vid manuell granskning kunde flera falska positiva träffar
identifieras, exempelvis när bokstavskombinationen förekom inuti andra ord.

Baselinen justerades därför till matchning med tydliga ordgränser.

### Riktiga baseline:

In [44]:
# förbättrad baseline: 
def baseline_extract_skills(text, skills):
    found_skills = []

    text = text.lower()

    for skill in skills:
        pattern = r"\b" + re.escape(skill.lower()) + r"\b"

        if re.search(pattern, text):
            found_skills.append(skill)

    return found_skills

In [45]:
# testar riktiga baselinen på samma 20 annonser:
sample_df["baseline_skills"] = sample_df["description"].apply(
    lambda text: baseline_extract_skills(text, skill_names)
)

In [46]:
sample_df[
    [
        "title",
        "naive_baseline_skills",
        "baseline_skills"
    ]
]

,title,naive_baseline_skills,baseline_skills
0,Business Consultant Dynamics 365 Supply Chain ...,[],[]
1,Hälsoinformatiker för framtidens vårdinformation,"[Informationsmodellering, Förändringsledning, ...","[Informationsmodellering, Förändringsledning, ..."
2,Data Scientist till Innovativt Bolag,[REST],[]
3,Dynamics 365 Finance Consultant,[],[]
4,"Enterprise arkitekt, Gävle/Uppsala/Stockholm",[],[]
5,IT-konsult,[],[]
6,Senior konsult inom D365 Finance & Operations ...,[],[]
7,Senior OpenStack-arkitekt,[REST],[]
8,Lösningsarkitekt,[Webbutveckling],[Webbutveckling]
9,Senior system architect,[],[]


In [47]:
# kör riktiga baselinen på hela train_set:
df["baseline_skills"] = df["description"].apply(
    lambda text: baseline_extract_skills(text, skill_names)
)

In [48]:
# räknar kompetenser för den slutliga baselinen:
all_baseline_skills = df["baseline_skills"].explode()

baseline_skill_counts = all_baseline_skills.value_counts()

baseline_skill_counts.head(20)

baseline_skills
Informationsarkitektur     136
Förändringsledning         130
Datamodellering            116
Informationsmodellering     92
REST                        68
Masterdata                  66
Informatik                  60
Systemförvaltning           59
Agila arbetsmetoder         54
Verksamhetsanalys           38
API-design                  17
Projektledning, IT          15
Prediktiv analys            11
Webbutveckling               8
Nätverksteknik               8
Datawarehouse                7
Interaktionsdesign           5
Databasdesign                4
Budgetansvar                 4
Molnteknik                   4
Name: count, dtype: int64

In [49]:
# jämför de två versionerna: 
comparison = pd.DataFrame({
    "naive_baseline": skill_counts,
    "baseline": baseline_skill_counts
}).fillna(0)

comparison.head(20)

,naive_baseline,baseline
baseline_skills,,
API-design,17,17.0
Agila arbetsmetoder,54,54.0
Automationsteknik,1,1.0
Budgetansvar,4,4.0
Databasdesign,4,4.0
Datamodellering,119,116.0
Datawarehouse,9,7.0
Design Thinking,2,2.0
Förändringsledning,130,130.0


Den ursprungliga naiva baselinen använde enkel substring-matchning, vilket gav falska positiva träffar. Ett tydligt exempel var kompetensen REST, som bland annat matchades i ordet "prestigelös" och en del av andra ord. 
Efter att matchningen förbättrades med reguljära uttryck och ordgränser minskade REST från 798 till 68 träffar, medan flera andra kompetenser behöll samma antal träffar. Detta tyder på att den förbättrade baselinen minskar antalet falska positiva matchningar.

Den justerade versionen används därför som projektets baseline.

## Sammanfattning av baseline

Projektets baseline bygger på regelbaserad matchning mellan kompetenser från
Arbetsförmedlingens SSYK 2511-taxonomi och jobbannonsernas beskrivningar.

Den första naiv implementation använde substring-matchning. Den användes för
att identifiera problem med falska positiva träffar, exempelvis när korta
kompetenstermer förekom som delar av andra ord.

Den slutliga baselinen använder därför tydligare ordavgränsning vid
matchningen. Metoden fungerar som en enkel referensmetod men är fortfarande
begränsad eftersom den kan missa exempelvis förkortningar, synonymer och
alternativa formuleringar.

Baselinen och resultatet kommer därför senare att utvecklas på träningsdata och utvärderas på det
separata manuellt annoterade test-setet tillsammans med projektets NLP-metoder.